# 01 – Exploratory Data Analysis: Credit Card Fraud Detection

**Projekt:** `autoencoder-fraud-detection`

**Mål:** Bygga en Autoencoder som lär sig hur legitima transaktioner ser ut, och som därmed kan flagga bedrägliga transaktioner genom att de får ett högt rekonstruktionsfel (modellen "känner inte igen" dem).

**Dataset:** Kaggle *"Credit Card Fraud Detection"* (`mlg-ulb/creditcardfraud`)
- 284 807 transaktioner, varav 492 är bedrägerier (~0,17 %)
- 28 PCA-transformerade features (`V1`–`V28`), samt `Time`, `Amount` och `Class` (otransformerade)
- `Class`: 0 = legitim, 1 = bedrägeri

**Den här notebooken:** laddar in datasetet, kontrollerar datakvalitet, utforskar klassobalans och variablernas fördelning, och sparar figurer/rapporter som vi återanvänder i presentationen senare.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

# Skapa output-mappar om de inte redan finns
os.makedirs('../reports/figures', exist_ok=True)

RANDOM_STATE = 42


## Ladda in dataset

Vi läser in `creditcard.csv` och tittar på grundläggande struktur: antal rader/kolumner, datatyper och de första raderna.

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')

print(f"Antal rader: {df.shape[0]}")
print(f"Antal kolumner: {df.shape[1]}")
print()
df.info()


In [ ]:
df.head()


**Vad ser vi?**

*(Fylls i utifrån den faktiska outputen ovan när du kört cellerna – kommenterar antal rader/kolumner, datatyper och om något ser avvikande ut.)*

## Saknade värden och dubbletter

Innan vi går vidare kontrollerar vi grundläggande datakvalitet: saknade värden och dubblettrader.

In [ ]:
print("Saknade värden totalt:", df.isnull().sum().sum())
print("Antal dubblettrader:", df.duplicated().sum())


**Vad ser vi?**

*(Fylls i utifrån outputen – om datasetet är rent kan vi gå vidare utan extra städning; annars noterar vi vad som behöver hanteras.)*

## Klassfördelning – legitima vs bedrägliga transaktioner

Det här är den centrala egenskapen hos datasetet och anledningen till att vi väljer en Autoencoder-ansats: klasserna är extremt obalanserade, så vi tränar modellen enbart på legitima transaktioner och låter den flagga det som avviker.

In [ ]:
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print("Antal per klass:")
print(class_counts)
print()
print("Andel per klass (%):")
print(class_pct.round(4))

fig, ax = plt.subplots(figsize=(6, 4))
labels = class_counts.index.map({0: 'Legitim', 1: 'Bedrägeri'})
sns.barplot(x=labels, y=class_counts.values, ax=ax)
ax.set_title('Klassfördelning: Legitima vs Bedrägliga transaktioner')
ax.set_ylabel('Antal transaktioner')
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 3000, f"{v:,}", ha='center')
plt.tight_layout()
plt.savefig('../reports/figures/01_class_distribution.png', dpi=150)
plt.show()

# Spara sammanfattning till rapport (återanvänds i presentationen)
class_summary = pd.DataFrame({'count': class_counts, 'percentage': class_pct.round(4)})
class_summary.to_csv('../reports/class_distribution.csv')


**Vad ser vi?**

*(Fylls i utifrån outputen – exakt antal/andel bedrägerier, och vad det innebär för hur vi bör utvärdera modellen senare, t.ex. varför "accuracy" ensamt blir missvisande.)*

## Fördelning av `Time` och `Amount`

`Time` och `Amount` är de enda features som inte är PCA-transformerade. Vi tittar på deras fördelningar innan skalning.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Time'], bins=50, color='steelblue')
axes[0].set_title('Fördelning av Time (sekunder sedan första transaktionen)')
axes[0].set_xlabel('Time (sekunder)')
axes[0].set_ylabel('Antal transaktioner')

axes[1].hist(df['Amount'], bins=50, color='darkorange')
axes[1].set_title('Fördelning av Amount')
axes[1].set_xlabel('Amount')
axes[1].set_yscale('log')
axes[1].set_ylabel('Antal transaktioner (log-skala)')

plt.tight_layout()
plt.savefig('../reports/figures/02_time_amount_distribution.png', dpi=150)
plt.show()


**Vad ser vi?**

*(Fylls i utifrån outputen – t.ex. om Time visar tydliga dygnsmönster, och hur skevt Amount är fördelat.)*

## Amount per klass

Vi jämför `Amount` mellan legitima och bedrägliga transaktioner – om bedrägerier tenderar att ha andra beloppsmönster är det bra att känna till innan vi tränar modellen.

In [ ]:
amount_by_class = df.groupby('Class')['Amount'].describe()
print(amount_by_class)
amount_by_class.to_csv('../reports/amount_by_class_summary.csv')

fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(x='Class', y='Amount', data=df, ax=ax)
ax.set_yscale('log')
ax.set_xticklabels(['Legitim', 'Bedrägeri'])
ax.set_title('Amount per klass (log-skala)')
plt.tight_layout()
plt.savefig('../reports/figures/03_amount_by_class_boxplot.png', dpi=150)
plt.show()


**Vad ser vi?**

*(Fylls i utifrån outputen – skillnader i medel/median/spridning mellan legitima och bedrägliga belopp.)*

## Korrelation mellan features och `Class`

`V1`–`V28` är redan PCA-transformerade och därmed inbördes okorrelerade, men vi kan ändå titta på vilka som korrelerar mest (positivt/negativt) med `Class`. Det ger en känsla för vilka mönster autoencodern behöver kunna återskapa väl för legitima transaktioner.

In [ ]:
correlations = df.corr()['Class'].drop('Class').sort_values()

fig, ax = plt.subplots(figsize=(8, 10))
colors = correlations.apply(lambda x: 'crimson' if x < 0 else 'seagreen')
correlations.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Korrelation mellan features och Class')
ax.set_xlabel('Korrelationskoefficient')
plt.tight_layout()
plt.savefig('../reports/figures/04_feature_correlation_with_class.png', dpi=150)
plt.show()

correlations.to_csv('../reports/feature_correlation_with_class.csv', header=['correlation'])


**Vad ser vi?**

*(Fylls i utifrån outputen – vilka V-features som korrelerar starkast med bedrägeri, i vilken riktning.)*

## Sammanfattning av EDA

*(Fylls i utifrån resultaten ovan – de viktigaste insikterna vi tar med oss till modelleringssteget, t.ex. klassobalansens storlek, behov av skalning av Time/Amount, och eventuella mönster i vilka features som skiljer bedrägerier från legitima transaktioner.)*

**Nästa steg:** `02_autoencoder_model.ipynb` – förbehandling, träning av autoencoder på legitima transaktioner, och tröskelbestämning.